# Анализ визитов, регистраций и рекламы

Данные: API (визиты, регистрации) + `ads.csv` (реклама). Результат: `conversion.json`, `ads.json`, графики в `charts/`.

In [ ]:
# Ячейка 1 — настройки и загрузка данных
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()  # подхватываем переменные из файла .env

API_URL = os.getenv('API_URL')
DATE_BEGIN = os.getenv('DATE_BEGIN')
DATE_END = os.getenv('DATE_END')

print(f'API: {API_URL}, период: {DATE_BEGIN} — {DATE_END}')

# Визиты и регистрации — из API
resp = requests.get(f'{API_URL}/visits',
                    params={'begin': DATE_BEGIN, 'end': DATE_END})
resp.raise_for_status()
data = resp.json()

resp_reg = requests.get(f'{API_URL}/registrations',
                        params={'begin': DATE_BEGIN, 'end': DATE_END})
resp_reg.raise_for_status()
data_reg = resp_reg.json()

# Реклама — из файла в корне проекта (закоммичен в репозиторий)
ads = pd.read_csv('./ads.csv')

visits_df = pd.DataFrame(data)
regs_df = pd.DataFrame(data_reg)

print('Визитов получено:', len(visits_df))
print('Регистраций получено:', len(regs_df))
print('Записей о рекламе:', len(ads))

In [ ]:
# Ячейка 2 — инспекция сырых данных
print('Визиты:', visits_df.shape)
print(visits_df.columns.tolist())

print('\nРегистрации:', regs_df.shape)
print(regs_df.columns.tolist())

print('\nРеклама:')
print(ads.columns.tolist())
print(ads.head(3))

In [ ]:
# Ячейка 3 — расчёт конверсии по дням и платформам

# 1. Переименования
visits = visits_df.rename(columns={'visit_id': 'uuid', 'datetime': 'date'})
visits['date_dt'] = pd.to_datetime(visits['date'])

regs = regs_df.rename(columns={'datetime': 'date'})
regs['date_dt'] = pd.to_datetime(regs['date'])

# 2. Убираем ботов (по user_agent и platform)
visits_clean = visits[
    (~visits['user_agent'].str.contains('bot', case=False, na=False))
    & (visits['platform'] != 'bot')
].copy()
print(f'Визитов после удаления ботов: {len(visits_clean)} (было {len(visits)})')

# 3. Берём только ПОСЛЕДНИЙ визит для каждого visit_id
visits_clean = visits_clean.sort_values('date_dt')
visits_last = visits_clean.drop_duplicates(subset='uuid', keep='last')
print(f'Визитов после удаления повторных заходов: {len(visits_last)}')

# 4. Агрегация по дню и платформе
visits_agg = (
    visits_last
    .assign(date_group=visits_last['date_dt'].dt.floor('D'))
    .groupby(['date_group', 'platform'], as_index=False)
    .size()
    .rename(columns={'size': 'visits'})
)

regs_agg = (
    regs
    .assign(date_group=regs['date_dt'].dt.floor('D'))
    .groupby(['date_group', 'platform'], as_index=False)
    .size()
    .rename(columns={'size': 'registrations'})
)

# 5. Склейка и конверсия
conversion = visits_agg.merge(regs_agg, on=['date_group', 'platform'], how='left')
conversion['registrations'] = conversion['registrations'].fillna(0).astype(int)
conversion['conversion'] = conversion['registrations'] / conversion['visits'] * 100
conversion = conversion.sort_values(['date_group', 'platform']).reset_index(drop=True)

# 6. Сохраняем: даты — строками в ISO-формате
conversion.to_json('./conversion.json', date_format='iso')

print('Строк в conversion:', len(conversion))
print('Сумма визитов (уникальных):', conversion['visits'].sum())
print('Сумма регистраций:', conversion['registrations'].sum())
print(conversion.head())

In [ ]:
# Ячейка 4 — объединение конверсии с данными о рекламе

# 1. Реклама: дата -> день, агрегируем
ads['date_dt'] = pd.to_datetime(ads['date'])
ads['date_group'] = ads['date_dt'].dt.floor('D')

ads_agg = (
    ads
    .groupby('date_group', as_index=False)
    .agg(cost=('cost', 'sum'),
         utm_campaign=('utm_campaign', 'first'))
)

# 2. Конверсия по датам (без платформ)
daily = (
    conversion
    .groupby('date_group', as_index=False)
    .agg(visits=('visits', 'sum'),
         registrations=('registrations', 'sum'))
)

# 3. Объединяем: дни без рекламы — cost=0, utm_campaign='none'
ads_df = daily.merge(ads_agg, on='date_group', how='left')
ads_df['cost'] = ads_df['cost'].fillna(0)
ads_df['utm_campaign'] = ads_df['utm_campaign'].fillna('none')
ads_df = ads_df.sort_values('date_group').reset_index(drop=True)

ads_df.to_json('./ads.json', date_format='iso')

print('Строк в ads_final:', len(ads_df))
print('Общая стоимость рекламы:', ads_df['cost'].sum())
print('Дней с рекламой:', (ads_df['cost'] > 0).sum())
print(ads_df.head(10))

In [ ]:
# Ячейка 5 — подготовка данных для графиков

daily_cr = daily.copy().set_index('date_group')
daily_cr['conversion'] = (daily_cr['registrations'] / daily_cr['visits'] * 100).round(2)

conv_pivot = conversion.pivot_table(index='date_group', columns='platform',
                                    values=['visits', 'registrations'], aggfunc='sum')
visits_by_platform = conv_pivot['visits']
regs_by_platform = conv_pivot['registrations']
cr_by_platform = (regs_by_platform / visits_by_platform * 100).round(2)

# Периоды рекламных кампаний
campaign_periods = []
for _, row in ads_df.sort_values('date_group').iterrows():
    if row['utm_campaign'] != 'none':
        if campaign_periods and campaign_periods[-1]['campaign'] == row['utm_campaign'] \
           and (row['date_group'] - campaign_periods[-1]['end']).days == 1:
            campaign_periods[-1]['end'] = row['date_group']
        else:
            campaign_periods.append({'campaign': row['utm_campaign'],
                                     'start': row['date_group'],
                                     'end': row['date_group']})

print('Кампаний найдено:', len(campaign_periods))
for c in campaign_periods:
    print(f"  {c['campaign']}: {c['start'].date()} — {c['end'].date()}")

In [ ]:
# Ячейка 6 — все графики проекта
import os
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

os.makedirs('./charts', exist_ok=True)

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['font.size'] = 11

platform_colors = {'web': '#1f77b4', 'android': '#2ca02c', 'ios': '#ff7f0e'}
platforms = ['web', 'android', 'ios']

# === 1. Итоговые визиты ===
fig, ax = plt.subplots()
ax.plot(daily_cr.index, daily_cr['visits'], color='#1f77b4', linewidth=1.5)
ax.set_title('Итоговые визиты по дням'); ax.set_xlabel('Дата'); ax.set_ylabel('Визиты')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m'))
fig.autofmt_xdate(); plt.tight_layout()
plt.savefig('./charts/visits.png'); plt.show()

# === 2. Визиты по платформам ===
fig, ax = plt.subplots()
for p in platforms:
    ax.plot(visits_by_platform.index, visits_by_platform[p], label=p, color=platform_colors[p])
ax.set_title('Итоговые визиты по платформам'); ax.set_xlabel('Дата'); ax.set_ylabel('Визиты')
ax.legend(title='Платформа')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m'))
fig.autofmt_xdate(); plt.tight_layout()
plt.savefig('./charts/visits_by_platform.png'); plt.show()

# === 3. Итоговые регистрации ===
fig, ax = plt.subplots()
ax.plot(daily_cr.index, daily_cr['registrations'], color='#d62728', linewidth=1.5)
ax.set_title('Итоговые регистрации по дням'); ax.set_xlabel('Дата'); ax.set_ylabel('Регистрации')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m'))
fig.autofmt_xdate(); plt.tight_layout()
plt.savefig('./charts/registrations.png'); plt.show()

# === 4. Регистрации по платформам ===
fig, ax = plt.subplots()
for p in platforms:
    ax.plot(regs_by_platform.index, regs_by_platform[p], label=p, color=platform_colors[p])
ax.set_title('Регистрации по платформам'); ax.set_xlabel('Дата'); ax.set_ylabel('Регистрации')
ax.legend(title='Платформа')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m'))
fig.autofmt_xdate(); plt.tight_layout()
plt.savefig('./charts/registrations_by_platform.png'); plt.show()

# === 5. Конверсия по платформам ===
fig, ax = plt.subplots()
for p in platforms:
    ax.plot(cr_by_platform.index, cr_by_platform[p], label=p, color=platform_colors[p])
ax.set_title('Конверсия по платформам, %'); ax.set_xlabel('Дата'); ax.set_ylabel('Конверсия, %')
ax.legend(title='Платформа')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m'))
fig.autofmt_xdate(); plt.tight_layout()
plt.savefig('./charts/conversion_by_platform.png'); plt.show()

# === 6. Средняя конверсия ===
mean_cr = daily_cr['conversion'].mean()
fig, ax = plt.subplots()
ax.plot(daily_cr.index, daily_cr['conversion'], color='#9467bd', linewidth=1.5,
        label=f'Средняя конверсия: {mean_cr:.1f}%')
ax.axhline(mean_cr, color='#9467bd', linestyle='--', alpha=0.6)
ax.set_title('Средняя конверсия по дням'); ax.set_xlabel('Дата'); ax.set_ylabel('Конверсия, %')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m'))
fig.autofmt_xdate(); plt.tight_layout()
plt.savefig('./charts/average_conversion.png'); plt.show()
print(f'Средняя конверсия за период: {mean_cr:.2f}%')

# === 7. Затраты на рекламу ===
fig, ax = plt.subplots()
ax.bar(ads_df['date_group'], ads_df['cost'], color='#17becf', width=1.0, label='Затраты')
ax.set_title('Затраты на рекламу по дням'); ax.set_xlabel('Дата'); ax.set_ylabel('Затраты')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m'))
fig.autofmt_xdate(); plt.tight_layout()
plt.savefig('./charts/ads_costs.png'); plt.show()

# === 8. Визиты с выделением кампаний ===
cmap = plt.get_cmap('tab20')
campaign_colors = {c['campaign']: cmap(i % 20) for i, c in enumerate(campaign_periods)}

fig, ax = plt.subplots()
for c in campaign_periods:
    ax.axvspan(c['start'], c['end'] + pd.Timedelta(days=1),
               color=campaign_colors[c['campaign']], alpha=0.25, label=c['campaign'])
ax.plot(daily_cr.index, daily_cr['visits'], color='#1f77b4', linewidth=1.5, label='Визиты')
ax.set_title('Визиты за весь период с выделением рекламных кампаний')
ax.set_xlabel('Дата'); ax.set_ylabel('Визиты')
ax.legend(loc='upper left', fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m'))
fig.autofmt_xdate(); plt.tight_layout()
plt.savefig('./charts/visits_with_campaigns.png'); plt.show()

# === 9. Регистрации с выделением кампаний ===
fig, ax = plt.subplots()
for c in campaign_periods:
    ax.axvspan(c['start'], c['end'] + pd.Timedelta(days=1),
               color=campaign_colors[c['campaign']], alpha=0.25, label=c['campaign'])
ax.plot(daily_cr.index, daily_cr['registrations'], color='#d62728', linewidth=1.5, label='Регистрации')
ax.set_title('Регистрации за весь период с выделением рекламных кампаний')
ax.set_xlabel('Дата'); ax.set_ylabel('Регистрации')
ax.legend(loc='upper left', fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m'))
fig.autofmt_xdate(); plt.tight_layout()
plt.savefig('./charts/registrations_with_campaigns.png'); plt.show()

In [ ]:
# Ячейка 7 — анализ для презентации

# 1. Влияние кампаний: средние визиты в дни рекламы vs без
campaign_days = ads_df[ads_df['utm_campaign'] != 'none']['date_group']
mask = daily_cr.index.isin(campaign_days)

print('=== Визиты: дни с рекламой vs без ===')
print(f'Среднее в дни кампаний: {daily_cr.loc[mask, "visits"].mean():.0f}')
print(f'Среднее вне кампаний:   {daily_cr.loc[~mask, "visits"].mean():.0f}')

# 2. Каждая кампания отдельно: до / во время
for c in campaign_periods:
    start, end = c['start'], c['end']
    before = daily_cr.loc[(daily_cr.index >= start - pd.Timedelta(days=14)) &
                          (daily_cr.index < start), 'visits'].mean()
    during = daily_cr.loc[(daily_cr.index >= start) &
                          (daily_cr.index <= end), 'visits'].mean()
    cost = ads_df.loc[ads_df['utm_campaign'] == c['campaign'], 'cost'].sum()
    print(f"{c['campaign']}: было {before:.0f}/день, стало {during:.0f}/день "
          f"({(during/before-1)*100:+.0f}%), бюджет {cost:.0f}")

# 3. Корреляция затрат и визитов
corr = daily_cr['visits'].corr(ads_df.set_index('date_group')['cost'])
print(f'\nКорреляция затрат и визитов: {corr:.2f}')

# 4. Просадки визитов: ниже 70% скользящего среднего за 14 дней
rolling = daily_cr['visits'].rolling(14).mean().shift(1)
drops_v = daily_cr[daily_cr['visits'] < rolling * 0.7]
print('\n=== Просадки визитов (>30% ниже нормы) ===')
print(drops_v[['visits']].round(0).to_string())

# 5. Средняя конверсия по платформам за весь период
print('\n=== Средняя конверсия по платформам, % ===')
print(cr_by_platform.mean().round(2).to_string())